In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import urllib.request, re, json, csv, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import fisher_exact
warnings.filterwarnings('ignore')

GRAMMAR_BASE = "https://raw.githubusercontent.com/NephVar/NephVar/main/grammars"
CLINVAR_BASE = "https://raw.githubusercontent.com/Joshua-Pillai/NephVar/main"

IDR_GENES = {
    "CGN":   ["CD46","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["BMP4","CHD1L","EYA1","GATA3","HNF1B","MUC1","PAX2","ROBO2","SALL1",
               "SIX1","SIX2","SIX5","SOX17","SRGAP1","TBX18","TNXB","UPK3A","CHRM3",
               "FREM2","GRIP1","LRIG2","KAL1"],
    "SRNS":  ["ADCK4","ARHGDIA","CD2AP","FAT1","ITGA3","ITGB4","KANK1","KANK2","KANK4",
               "LAMB2","MYO1E","NPHS1","NPHS2","NUP107","PLCE1","SMARCAL1","ANLN",
               "ARHGAP24","INF2","LMX1B","MYH9","WT1"],
    "USD":   ["ATP6V0A4","CA2","CASR","CLDN19","FAM20A","HNF4A","OCRL","SLC12A1",
               "SLC2A9","SLC3A1","SLC4A1","SLC9A3R1","VDR"],
    "NPHP":  ["NPHP1","INVS","NPHP3","NPHP4","CEP290","GLIS2","RPGRIP1L","SDCCAG8",
               "ZNF423","CEP164","ANKS6"],
}

PATHOGENIC = {"Pathogenic","Likely pathogenic","Pathogenic/Likely pathogenic",
              "Pathogenic, low penetrance","Likely pathogenic, low penetrance",
              "Pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Pathogenic, low penetrance"}
VUS_SET    = {"Uncertain significance","Uncertain significance/Uncertain risk allele",
              "Uncertain risk allele","Likely pathogenic/Likely risk allele",
              "Likely risk allele","Uncertain significance/VUS-mid"}
BENIGN     = {"Benign","Likely benign","Benign/Likely benign"}

def bucket(gc):
    gc = gc.strip()
    if gc in PATHOGENIC: return "P/LP"
    if gc in VUS_SET:    return "VUS"
    if gc in BENIGN:     return "B/LB"
    return "Other"

PANEL_ORDER = ["CGN","CAKUT","SRNS","USD","NPHP"]
print("Setup complete.")

Setup complete.


In [2]:
# ── Cell 2: Scrape IDR boundaries from NephVar/NephVar grammars pages ──────────
idr_records = []
for panel, genes in IDR_GENES.items():
    for gene in genes:
        url = f"{GRAMMAR_BASE}/{gene}.html"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                html = r.read().decode('utf-8', errors='replace')
            m = re.search(r'const idrs\s*=\s*(\[.*?\]);', html)
            if m:
                for i, idr in enumerate(json.loads(m.group(1))):
                    idr_records.append({
                        "panel":        panel,
                        "gene":         gene,
                        "idr_num":      i + 1,
                        "idr_start":    idr["start"],
                        "idr_end":      idr["end"],
                        "idr_len":      idr["end"] - idr["start"] + 1,
                        "cluster":      idr["cluster"],
                        "cluster_desc": idr.get("cluster_desc", ""),
                        "cluster_color":idr.get("cluster_color", "#888"),
                    })
            else:
                print(f"  WARNING: no IDRs found for {gene}")
        except Exception as e:
            print(f"  ERROR {gene}: {e}")

df_idr = pd.DataFrame(idr_records)

# Build per-gene interval lookup
gene_idrs = {}
for _, row in df_idr.iterrows():
    gene_idrs.setdefault(row['gene'], []).append((row['idr_start'], row['idr_end']))

print(f"IDR records:  {len(df_idr)}")
print(f"Genes with IDRs: {df_idr['gene'].nunique()}")
print()
print(df_idr.groupby('panel')[['gene','idr_num']].agg(
    genes=('gene','nunique'), idrs=('idr_num','count')
).loc[PANEL_ORDER])

IDR records:  198
Genes with IDRs: 74

       genes  idrs
panel             
CGN        6    18
CAKUT     22    67
SRNS      22    61
USD       13    16
NPHP      11    36


In [5]:
# ── Cell 3: Load full NephVar dataset, dedup globally, then subset to IDR genes ──
import urllib.request, io
import pandas as pd

# Full gene panels (all NephVar genes, not just the IDR-annotated subset)
ALL_PANELS = {
    "CGN":   ["C3","CD46","CFH","CFHR5","CFI","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["ACE","AGT","AGTR1","BMP4","CHD1L","CHRM3","DSTYK","EYA1","FGF20","FRAS1",
              "FREM1","FREM2","GATA3","GRIP1","HNF1B","HPSE2","ITGA8","KAL1","LRIG2","MUC1",
              "PAX2","REN","RET","ROBO2","SALL1","SIX1","SIX2","SIX5","SOX17","SRGAP1",
              "TBX18","TNXB","TRAP1","UMOD","UPK3A","WNT4"],
    "SRNS":  ["ACTN4","ADCK4","ANLN","ARHGAP24","ARHGDIA","CD2AP","COQ2","COQ6","CRB2",
              "CUBN","DGKE","EMP2","FAT1","INF2","ITGA3","ITGB4","KANK1","KANK2","KANK4",
              "LAMB2","LMX1B","MTTL1","MYH9","MYO1E","NPHS1","NPHS2","NUP107","NUP205",
              "NUP93","PDSS2","PLCE1","PTPRO","SCARB2","SMARCAL1","TRPC6","WDR73","WT1","XPO5"],
    "USD":   ["ADCY10","AGXT","APRT","ATP6V0A4","ATP6V1B1","CA2","CASR","CLCN5","CLCNKB",
              "CLDN16","CLDN19","CYP24A1","FAM20A","GRHPR","HNF4A","HOGA1","HPRT1","KCNJ1",
              "OCRL","SLC12A1","SLC22A12","SLC2A9","SLC34A1","SLC34A3","SLC3A1","SLC4A1",
              "SLC7A9","SLC9A3R1","VDR","XDH"],
    "NPHP":  ["ANKS6","CEP164","CEP290","GLIS2","INVS","IQCB1","NEK8","NPHP1","NPHP3",
              "NPHP4","RPGRIP1L","SDCCAG8","TMEM67","TTC21B","WDR19","ZNF423"],
}

all_rows = []
for panel, genes in ALL_PANELS.items():
    for gene in genes:
        url = f"{CLINVAR_BASE}/{panel}/{gene}.txt"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                content = r.read().decode('utf-8', errors='replace')
            df = pd.read_csv(io.StringIO(content), sep='\t', low_memory=False)
            df['gene']  = gene
            df['panel'] = panel
            all_rows.append(df)
        except Exception as e:
            print(f"  ERROR {gene}: {e}")

df_all = pd.concat(all_rows, ignore_index=True)
df_all.columns = [c.strip() for c in df_all.columns]
print(f"Total raw rows (all panels, all genes): {len(df_all):,}")

# Global dedup on VariationID — this is the verified 117,373 baseline
df_global = df_all.drop_duplicates(subset='VariationID', keep='first').copy()
print(f"Total unique VariationIDs (global): {len(df_global):,}")
assert len(df_global) == 117373, f"Expected 117,373, got {len(df_global):,} — check panel gene lists"

# Now subset to only the genes that have IDR annotations
idr_gene_set = {g for genes in IDR_GENES.values() for g in genes}
df_cv = df_global[df_global['gene'].isin(idr_gene_set)].copy()

print(f"\nIDR-annotated genes: {len(idr_gene_set)}")
print(f"Unique variants among IDR genes (post-global-dedup): {len(df_cv):,}")
print()
print(df_cv.groupby('panel').size().reindex(PANEL_ORDER).rename('rows'))

Total raw rows (all panels, all genes): 119,315
Total unique VariationIDs (global): 117,373

IDR-annotated genes: 74
Unique variants among IDR genes (post-global-dedup): 75,643

panel
CGN      13034
CAKUT    14967
SRNS     21598
USD       9911
NPHP     16133
Name: rows, dtype: int64


In [7]:
# ── Cell 4: Filter missense, parse residue position, apply ACMG bucket ─────────
df_miss = df_cv[
    df_cv['Molecular consequence'].str.contains('missense', case=False, na=False)
].copy()

def parse_residue(pc):
    if not pc or pd.isna(pc): return None
    s = str(pc).split(';')[0].strip().replace('p.', '')
    m = re.search(r'[A-Za-z*]+([0-9]+)[A-Za-z*=]', s)
    return int(m.group(1)) if m else None

df_miss['residue'] = df_miss['Protein change'].apply(parse_residue)
df_miss['acmg']    = df_miss['Germline classification'].apply(bucket)

parsed = df_miss['residue'].notna().sum()
total  = len(df_miss)
print(f"Total missense:      {total}")
print(f"Residue parsed:      {parsed} ({100*parsed/total:.1f}%)")
print(f"Unparseable:         {total - parsed}")
print()
print("ACMG distribution:")
print(df_miss['acmg'].value_counts().to_string())

Total missense:      32081
Residue parsed:      32063 (99.9%)
Unparseable:         18

ACMG distribution:
acmg
VUS      27657
P/LP      2460
B/LB      1962
Other        2


In [8]:
# ── Cell 5: Map each missense variant to IDR or non-IDR ───────────────────────
def in_idr(gene, residue):
    if residue is None: return None
    for (s, e) in gene_idrs.get(gene, []):
        if s <= residue <= e: return True
    return False

df_miss['in_idr'] = df_miss.apply(
    lambda r: in_idr(r['gene'], r['residue']), axis=1
)

# Restrict to variants with parseable residue
df_a = df_miss[df_miss['residue'].notna()].copy()

n_idr = df_a['in_idr'].sum()
n_tot = len(df_a)
print(f"Analyzable missense: {n_tot}")
print(f"In IDR:              {int(n_idr)} ({100*n_idr/n_tot:.1f}%)")
print(f"Not in IDR:          {int(n_tot - n_idr)} ({100*(n_tot-n_idr)/n_tot:.1f}%)")
print()
print("By panel:")
for panel in PANEL_ORDER:
    sub = df_a[df_a['panel'] == panel]
    n   = len(sub)
    ni  = sub['in_idr'].sum()
    print(f"  {panel:<8}  {int(ni):>5} / {n:>6} in IDR  ({100*ni/n:.1f}%)")

Analyzable missense: 32063
In IDR:              8337 (26.0%)
Not in IDR:          23726 (74.0%)

By panel:
  CGN        3194 /   4836 in IDR  (66.0%)
  CAKUT      1540 /   7061 in IDR  (21.8%)
  SRNS       1919 /   9373 in IDR  (20.5%)
  USD         508 /   4458 in IDR  (11.4%)
  NPHP       1176 /   6335 in IDR  (18.6%)


In [9]:
# ── Cell 6: Panel summary table ────────────────────────────────────────────────
print(f"{'Panel':<8} {'Genes':>6} {'Missense':>9} {'In IDR':>8} {'IDR%':>7}  "
      f"{'IDR dominant':>20}  {'non-IDR dominant':>20}")
print("-" * 80)

for panel in PANEL_ORDER:
    sub     = df_a[df_a['panel'] == panel]
    n_total = len(sub)
    n_idr   = sub['in_idr'].sum()
    pct_idr = 100 * n_idr / n_total if n_total else 0
    idr_sub = sub[sub['in_idr']]
    non_sub = sub[~sub['in_idr']]

    def top(s):
        if len(s) == 0: return "—"
        vc = s['acmg'].value_counts(normalize=True)
        return f"{vc.index[0]} ({100*vc.iloc[0]:.0f}%)"

    print(f"{panel:<8} {len(IDR_GENES[panel]):>6} {n_total:>9} {int(n_idr):>8} "
          f"{pct_idr:>6.1f}%  {top(idr_sub):>20}  {top(non_sub):>20}")

Panel     Genes  Missense   In IDR    IDR%          IDR dominant      non-IDR dominant
--------------------------------------------------------------------------------
CGN           6      4836     3194   66.0%             VUS (53%)             VUS (87%)
CAKUT        22      7061     1540   21.8%             VUS (91%)             VUS (89%)
SRNS         22      9373     1919   20.5%             VUS (87%)             VUS (87%)
USD          13      4458      508   11.4%             VUS (94%)             VUS (86%)
NPHP         11      6335     1176   18.6%             VUS (95%)             VUS (97%)


In [10]:
# ── Cell 7: Pathogenicity breakdown — IDR vs non-IDR per panel ─────────────────
print(f"{'Panel':<8} {'Region':<10} {'N':>6}  "
      f"{'P/LP':>5} {'P/LP%':>6}  {'VUS':>6} {'VUS%':>6}  {'B/LB':>5} {'B/LB%':>6}")
print("-" * 72)

for panel in PANEL_ORDER:
    sub = df_a[df_a['panel'] == panel]
    for region, mask in [("IDR", sub['in_idr']), ("non-IDR", ~sub['in_idr'])]:
        s   = sub[mask]
        n   = len(s)
        plp = (s['acmg'] == 'P/LP').sum()
        vus = (s['acmg'] == 'VUS').sum()
        blb = (s['acmg'] == 'B/LB').sum()
        print(f"{panel:<8} {region:<10} {n:>6}  "
              f"{plp:>5} {100*plp/n if n else 0:>5.1f}%  "
              f"{vus:>6} {100*vus/n if n else 0:>5.1f}%  "
              f"{blb:>5} {100*blb/n if n else 0:>5.1f}%")
    print()

Panel    Region          N   P/LP  P/LP%     VUS   VUS%   B/LB  B/LB%
------------------------------------------------------------------------
CGN      IDR          3194   1276  39.9%    1699  53.2%    219   6.9%
CGN      non-IDR      1642     99   6.0%    1423  86.7%    120   7.3%

CAKUT    IDR          1540     18   1.2%    1397  90.7%    125   8.1%
CAKUT    non-IDR      5521    199   3.6%    4940  89.5%    380   6.9%

SRNS     IDR          1919     33   1.7%    1677  87.4%    209  10.9%
SRNS     non-IDR      7454    380   5.1%    6497  87.2%    577   7.7%

USD      IDR           508      5   1.0%     475  93.5%     28   5.5%
USD      non-IDR      3950    397  10.1%    3415  86.5%    138   3.5%

NPHP     IDR          1176      3   0.3%    1123  95.5%     50   4.3%
NPHP     non-IDR      5159     49   0.9%    4994  96.8%    116   2.2%



In [11]:
# ── Cell 8: Fisher's exact test — P/LP and B/LB enrichment in IDR vs non-IDR ──
import scipy.stats as stats

def fisher_with_ci(a, b, c, d, confidence=0.95):
    """OR and 95% CI via Woolf logit method."""
    OR, pval = fisher_exact([[a, b], [c, d]], alternative='two-sided')
    try:
        log_or = np.log(OR)
        se     = np.sqrt(1/a + 1/b + 1/c + 1/d)
        z      = stats.norm.ppf(1 - (1 - confidence) / 2)
        ci_lo  = np.exp(log_or - z * se)
        ci_hi  = np.exp(log_or + z * se)
    except (ZeroDivisionError, ValueError):
        ci_lo, ci_hi = np.nan, np.nan
    return OR, pval, ci_lo, ci_hi

def fmt_pval(p):
    if p < 0.001:
        return f"{p:.2e}"
    elif p < 0.01:
        return f"{p:.4f}"
    else:
        return f"{p:.3f}"

fisher_results = {}

for panel in PANEL_ORDER:
    sub     = df_a[df_a['panel'] == panel]
    n_idr   = sub['in_idr'].sum()
    n_non   = (~sub['in_idr']).sum()

    print(f"{'─'*92}")
    print(f"  {panel}   (IDR n={int(n_idr)}, non-IDR n={int(n_non)}, total={int(n_idr+n_non)})")
    print(f"{'─'*92}")

    fisher_results[panel] = {}

    for label, target in [("P/LP", "P/LP"), ("B/LB", "B/LB")]:
        a = ( sub['in_idr'] &  (sub['acmg'] == target)).sum()  # IDR, target
        b = ( sub['in_idr'] & ~(sub['acmg'] == target)).sum()  # IDR, not target
        c = (~sub['in_idr'] &  (sub['acmg'] == target)).sum()  # non-IDR, target
        d = (~sub['in_idr'] & ~(sub['acmg'] == target)).sum()  # non-IDR, not target

        # Total subgroup sizes
        n_idr_target = a + b   # all IDR variants (denominator for IDR%)
        n_non_target = c + d   # all non-IDR variants (denominator for non-IDR%)
        n_target_tot = a + c   # total variants of this class across both regions

        OR, pval, ci_lo, ci_hi = fisher_with_ci(a, b, c, d)
        fisher_results[panel][label] = {
            'OR': OR, 'pval': pval, 'ci_lo': ci_lo, 'ci_hi': ci_hi,
            'a': a, 'b': b, 'c': c, 'd': d
        }

        idr_pct = 100 * a / n_idr_target if n_idr_target else 0
        non_pct = 100 * c / n_non_target if n_non_target else 0
        sig     = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))
        ci_str  = f"({ci_lo:.3f}–{ci_hi:.3f})"

        print(f"  {label:<6}  "
              f"IDR {a:>5}/{int(n_idr_target):>5} ({idr_pct:>5.1f}%)  "
              f"non-IDR {c:>5}/{int(n_non_target):>5} ({non_pct:>5.1f}%)  "
              f"[{target} total: {int(n_target_tot)}]  "
              f"OR={OR:>7.3f}  95% CI {ci_str:<20}  "
              f"p={fmt_pval(pval):>10}  {sig}")
    print()

────────────────────────────────────────────────────────────────────────────────────────────
  CGN   (IDR n=3194, non-IDR n=1642, total=4836)
────────────────────────────────────────────────────────────────────────────────────────────
  P/LP    IDR  1276/ 3194 ( 39.9%)  non-IDR    99/ 1642 (  6.0%)  [P/LP total: 1375]  OR= 10.369  95% CI (8.361–12.858)        p= 5.18e-160  ***
  B/LB    IDR   219/ 3194 (  6.9%)  non-IDR   120/ 1642 (  7.3%)  [B/LB total: 339]  OR=  0.934  95% CI (0.741–1.176)         p=     0.553  ns

────────────────────────────────────────────────────────────────────────────────────────────
  CAKUT   (IDR n=1540, non-IDR n=5521, total=7061)
────────────────────────────────────────────────────────────────────────────────────────────
  P/LP    IDR    18/ 1540 (  1.2%)  non-IDR   199/ 5521 (  3.6%)  [P/LP total: 217]  OR=  0.316  95% CI (0.195–0.514)         p=  9.96e-08  ***
  B/LB    IDR   125/ 1540 (  8.1%)  non-IDR   380/ 5521 (  6.9%)  [B/LB total: 505]  OR=  1.195

In [12]:
# ── Cell 9: Per-gene IDR missense breakdown ────────────────────────────────────
print(f"{'Panel':<8} {'Gene':<12} {'IDRs':>5} {'Missense':>9} {'In IDR':>7} {'IDR%':>6}  "
      f"{'P/LP(IDR)':>10}  {'VUS(IDR)':>9}  {'B/LB(IDR)':>10}")
print("-" * 95)

for panel in PANEL_ORDER:
    for gene in IDR_GENES[panel]:
        sub   = df_a[(df_a['panel'] == panel) & (df_a['gene'] == gene)]
        n     = len(sub)
        n_idr = sub['in_idr'].sum()
        if n == 0: continue

        idr_sub = sub[sub['in_idr']]
        n_i     = len(idr_sub)
        plp     = (idr_sub['acmg'] == 'P/LP').sum()
        vus     = (idr_sub['acmg'] == 'VUS').sum()
        blb     = (idr_sub['acmg'] == 'B/LB').sum()
        n_idrs  = len(gene_idrs.get(gene, []))
        pct     = 100 * n_idr / n if n else 0

        plp_str = f"{int(plp)} ({100*plp/n_i:.0f}%)" if n_i else "—"
        vus_str = f"{int(vus)} ({100*vus/n_i:.0f}%)" if n_i else "—"
        blb_str = f"{int(blb)} ({100*blb/n_i:.0f}%)" if n_i else "—"

        print(f"{panel:<8} {gene:<12} {n_idrs:>5} {n:>9} {int(n_idr):>7} {pct:>5.1f}%  "
              f"{plp_str:>10}  {vus_str:>9}  {blb_str:>10}")
    print()

Panel    Gene          IDRs  Missense  In IDR   IDR%   P/LP(IDR)   VUS(IDR)   B/LB(IDR)
-----------------------------------------------------------------------------------------------
CGN      CD46             1       270      29  10.7%      0 (0%)   27 (93%)      2 (7%)
CGN      COL4A3           3      1019     872  85.6%   327 (38%)  499 (57%)     46 (5%)
CGN      COL4A4           4      1036     885  85.4%   273 (31%)  569 (64%)     43 (5%)
CGN      COL4A5           1      1252    1113  88.9%   672 (60%)  340 (31%)    101 (9%)
CGN      COL4A6           8       328     255  77.7%      4 (2%)  225 (88%)    26 (10%)
CGN      FN1              1       931      40   4.3%      0 (0%)   39 (98%)      1 (2%)

CAKUT    BMP4             2       165      41  24.8%      2 (5%)   36 (88%)      3 (7%)
CAKUT    CHD1L            2       253      19   7.5%      0 (0%)   17 (89%)     2 (11%)
CAKUT    EYA1             2       233      85  36.5%      1 (1%)   74 (87%)    10 (12%)
CAKUT    GATA3         